# Retail Customers - Silver Layer Cleaning | Databricks End-to-End Project

In [0]:
customers_bronze_df = spark.table('retial_project.bronze.bronze_customers')
customers_bronze_df.display()

# Data Cleaning

##1. Trim Spaces

In [0]:

from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_time")
)

display(customers_df)


##2. Remove the duplicate Customers

In [0]:
customers_df.printSchema()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(desc("ingest_time"))
)

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = (
    customers_df
    .filter(col("row_num") == 1)
    .drop("row_num")
)

display(customers_df)

##3. Fix and Standardize Status

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

##4. Convert Loyalty_points to Integer

In [0]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "")
)

customers_df = customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").try_cast("int")
)
  

##5. Validate Email Format

In [0]:
customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True)

##6. Fixed Date formats

In [0]:
from pyspark.sql.functions import to_date, coalesce, try_to_date

customers_df = customers_df.withColumn(
    "dob_parsed",
    coalesce(
            try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
            try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
            try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
            try_to_date(col('date_of_birth'),'dd/MM/yyyy')
        )
)

customers_df = customers_df.withColumn(
    "registration_parsed",
    coalesce(
            try_to_date(col('registration_date'),'yyyy-MM-dd'),
            try_to_date(col('registration_date'),'dd-MM-yyyy'),
            try_to_date(col('registration_date'),'yyyy/MM/dd'),
            try_to_date(col('registration_date'),'dd/MM/yyyy')
        )
)

customers_df.display()

In [0]:
customers_df = customers_df.drop("date_of_birth", "registration_date") \
                           .withColumnRenamed("dob_parsed", "date_of_birth") \
                           .withColumnRenamed("registration_parsed", "registration_date")

##7. Standardize Gender

In [0]:
customers_df = customers_df.withColumn(
    "gender",
    upper(col("gender"))
)

customers_df = customers_df.withColumn(
    "gender",
    when(col("gender") == "MALE", "M")
    .when(col("gender") == "FEMALE", "F")
    .otherwise("U")
)

In [0]:

customers_df.write.format("delta").mode("overwrite").saveAsTable("retial_project.silver.silver_customers")